In [0]:
from pyspark.sql.functions import *

# ==========================================
# S3 PATHS
# ==========================================

incoming_path = "s3://workflowautomationgiri/incoming/"
processed_path = "s3://workflowautomationgiri/processed/"

# ==========================================
# GET ALL FILES FROM INCOMING
# ==========================================

all_files = dbutils.fs.ls(incoming_path)

csv_files = [
    file.path
    for file in all_files
    if file.path.endswith(".csv")
]

# ==========================================
# READ PROCESSED FILE LOG
# ==========================================

processed_df = spark.table(
    "workauto.gold.processed_files_log"
).filter(
    col("file_type") == "csv"
)

processed_files = [
    row["file_name"]
    for row in processed_df.collect()
]

# ==========================================
# FILTER ONLY NEW FILES
# ==========================================

new_files = []

for file in csv_files:

    file_name = file.split("/")[-1]

    if file_name not in processed_files:
        new_files.append(file)

# ==========================================
# PROCESS NEW FILES
# ==========================================

if len(new_files) > 0:

    # READ CSV FILES
    csv_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(new_files)
    )

    # ADD METADATA
    csv_df = (
        csv_df
        .withColumn(
            "source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "ingestion_time",
            current_timestamp()
        )
    )

    # APPEND TO BRONZE TABLE
    (
        csv_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.bronze.csv_raw")
    )

    # ======================================
    # UPDATE PROCESSED FILE LOG
    # ======================================

    processed_log_df = spark.createDataFrame(
        [
            (file.split("/")[-1], "csv")
            for file in new_files
        ],
        ["file_name", "file_type"]
    ).withColumn(
        "processed_time",
        current_timestamp()
    )

    (
        processed_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("workauto.gold.processed_files_log")
    )

    # ======================================
    # MOVE FILES TO PROCESSED FOLDER
    # ======================================

    for file in new_files:

        file_name = file.split("/")[-1]

        dbutils.fs.mv(
            file,
            processed_path + file_name
        )

    print("CSV Files Processed Successfully")

else:

    print("No New CSV Files Found")